# Export TorchScript — detecteur de symboles (RE-export autonome)

**Cas normal : tu n'as PAS besoin de ce notebook.** L'export est deja greffe a la
fin de `train_instance_seg.ipynb` (section 8) et se fait dans la meme session que
l'entrainement, sans aucun upload.

Ce notebook-ci sert seulement a **RE-exporter** un `model_final.pth` deja entraine
sans re-entrainer (si le code d'export doit etre corrige). Il lit le `.pth` depuis
**Google Drive** — surtout PAS via `files.upload()`, qui plafonne a ~1-2 Mo/s et fait
expirer la session sur un fichier de 352 Mo.

**Runtime CPU suffit** (Execution → Modifier le type d'execution → CPU) : l'export ne
touche pas le GPU, inutile d'attendre un GPU disponible.

Prerequis : depose `model_final.pth` et `coco_colab.zip` dans un dossier Drive
`MyDrive/comptage/` (l'upload vers Drive est resumable et se fait hors Colab).

## 1. Installer detectron2

In [ ]:
!pip -q install 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print('torch', torch.__version__, '| detectron2', detectron2.__version__)

## 2. Monter Google Drive (PAS d'upload navigateur)
Le `.pth` (352 Mo) est lu directement depuis Drive. Le zip sert uniquement a lire
l'ordre des NOMS de classes — **le meme zip que celui de l'entrainement**.
Ajuste `DRIVE_DIR` si tu as range les fichiers ailleurs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, shutil
DRIVE_DIR = '/content/drive/MyDrive/comptage'   # <-- ou tes fichiers sont ranges
shutil.copy(DRIVE_DIR + '/model_final.pth', 'model_final.pth')
shutil.unpack_archive(DRIVE_DIR + '/coco_colab.zip', '.')
CATS = sorted(json.load(open('coco_colab/train/annotations.json'))['categories'],
              key=lambda c: c['id'])
CLASS_NAMES = [c['name'] for c in CATS]
print(len(CLASS_NAMES), 'classes :', CLASS_NAMES)

## 3. Reconstruire l'archi + charger les poids + tracer
L'archi est reconstruite **exactement** comme a l'entrainement (yaml de base,
NUM_CLASSES, ancres). `SCORE_THRESH_TEST` est mis **bas (0.05)** : on veut TOUTES
les detections a l'export ; le filtrage par seuil-de-classe se fera en local
avec `thresholds.json`.

Taille d'entree **fixe 800x800** = la taille de tuile en production.

In [ ]:
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.export import TracingAdapter
import torch

NUM_CLASSES = len(CLASS_NAMES)
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    'COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml'))
cfg.MODEL.ROI_HEADS.NUM_CLASSES = NUM_CLASSES
# --- doit etre IDENTIQUE au run (cellule 10 de train_instance_seg.ipynb) ---
cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[8], [16], [32], [64], [128]]
cfg.TEST.DETECTIONS_PER_IMAGE = 400
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.05   # bas : filtrage par classe en local
cfg.MODEL.DEVICE = 'cpu'

model = build_model(cfg)
model.eval()
DetectionCheckpointer(model).load('model_final.pth')

# detectron2 attend list[dict] {'image': tensor CHW BGR 0-255, height, width}.
# La normalisation (PIXEL_MEAN/STD) est faite DANS le modele -> en local, fournir
# l'image BRUTE BGR 0-255 CHW.
example = torch.rand(3, 800, 800) * 255
inputs = [{'image': example, 'height': 800, 'width': 800}]

def inference(model, inputs):
    inst = model(inputs)[0]['instances']
    return [{'instances': inst}]

adapter = TracingAdapter(model, inputs, inference)
adapter.eval()
with torch.no_grad():
    ts = torch.jit.trace(adapter, adapter.flattened_inputs)
ts.save('model.ts')
print('TorchScript sauve -> model.ts')

## 4. Verifier la structure des sorties (CRUCIAL)
Le modele trace renvoie un **tuple de tenseurs a plat**. Le decodeur local doit
savoir lequel est quoi. Cette cellule fait passer une image de test et imprime la
forme + le type de chaque sortie. On s'attend a y trouver :

- un `[N, 4]` float = **boites** (x0,y0,x1,y1),
- un `[N]` float = **scores**,
- un `[N]` int = **indices de classe**,
- un `[N, 1, 800, 800]` ou `[N, 800, 800]` = **masques**,
- eventuellement un `[2]` = taille d'image.

**Copie-colle la sortie de cette cellule** : elle fixe l'ordre pour le decodeur local.

In [ ]:
import torch
ts = torch.jit.load('model.ts')
ts.eval()
test = torch.rand(3, 800, 800) * 255
with torch.no_grad():
    out = ts(test)
out = out if isinstance(out, (tuple, list)) else (out,)
print(f'{len(out)} sorties :')
for i, t in enumerate(out):
    if torch.is_tensor(t):
        print(f'  [{i}] shape={tuple(t.shape)}  dtype={t.dtype}')
    else:
        print(f'  [{i}] {type(t)} = {t}')

## 5. Telecharger `model.ts` + les metadonnees
`model_meta.json` porte l'ordre des classes et le format d'entree — le decodeur
local en a besoin.

In [ ]:
import json
meta = {
    'classes': CLASS_NAMES,
    'input_size': 800,
    'input_format': 'BGR_0_255_CHW',
    'note': 'normalisation faite DANS le modele ; fournir image BGR 0-255 CHW brute',
}
json.dump(meta, open('model_meta.json', 'w'), ensure_ascii=False, indent=1)
from google.colab import files
files.download('model.ts')
files.download('model_meta.json')
print('OK')